1. Steepest-Ascent Hill Climbing (Leo đồi dốc nhất)

In [6]:
import time
ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def manhattan_distance(current, goal):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = current[r][c]
            if val != 0:
                target_r, target_c = goal_pos[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def print_matrix(state):
    for row in state:
        print("  " + " ".join(str(x) if x != 0 else "_" for x in row))
    print()

def run_steepest_hill_climbing(start, goal, max_steps=1000):
    current = start
    path = []
    steps_count = 0
    curr_h = manhattan_distance(current, goal)

    while current != goal:
        if steps_count >= max_steps:
            return None, steps_count

        x, y = find_zero(current)
        best_neighbor = None
        best_h = curr_h
        best_move = None

        # Duyệt TẤT CẢ các hướng để tìm trạng thái có Heuristic thấp nhất
        for move, (dx, dy), _ in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                neighbor = swap(current, x, y, nx, ny)
                neighbor_h = manhattan_distance(neighbor, goal)

                if neighbor_h < best_h:
                    best_h = neighbor_h
                    best_neighbor = neighbor
                    best_move = move

        if best_neighbor is not None:
            current = best_neighbor
            curr_h = best_h
            path.append(best_move)
            steps_count += 1
        else:
            # Bị kẹt cục bộ
            return None, steps_count

    return path, steps_count

if __name__ == "__main__":

    START_STATE = ((1, 2, 3),
                   (4, 0, 6),
                   (7, 5, 8))

    GOAL_STATE  = ((1, 2, 3),
                   (4, 5, 6),
                   (7, 8, 0))

    print("Trạng thái Khởi đầu (START):")
    print_matrix(START_STATE)
    print("Trạng thái Đích (GOAL):")
    print_matrix(GOAL_STATE)
    print("-" * 60)

    algorithms = [
        ("Steepest-Ascent Hill Climbing", run_steepest_hill_climbing)
    ]

    for name, algo_func in algorithms:
        start_time = time.time()
        path, nodes_visited = algo_func(START_STATE, GOAL_STATE)
        execution_time = (time.time() - start_time) * 1000 # tính bằng mili-giây

        print(f"▶ Thuật toán: {name}")
        if path is not None:
            print(f"  - Trạng thái: THÀNH CÔNG ")
            print(f"  - Chi phí đường đi: {len(path)} bước")
            print(f"  - Lộ trình: {' -> '.join(path)}")
        else:
            print(f"  - Trạng thái: THẤT BẠI (Kẹt cục bộ hoặc vượt quá giới hạn)")
        print(f"  - Số vòng lặp thực hiện: {nodes_visited}")
        print(f"  - Thời gian xử lý: {execution_time:.4f} ms")
        print("-" * 60)

Trạng thái Khởi đầu (START):
  1 2 3
  4 _ 6
  7 5 8

Trạng thái Đích (GOAL):
  1 2 3
  4 5 6
  7 8 _

------------------------------------------------------------
▶ Thuật toán: Steepest-Ascent Hill Climbing
  - Trạng thái: THÀNH CÔNG 
  - Chi phí đường đi: 2 bước
  - Lộ trình: D -> R
  - Số vòng lặp thực hiện: 2
  - Thời gian xử lý: 0.0000 ms
------------------------------------------------------------


2. Stochastic Hill Climbing (Leo đồi ngẫu nhiên)

In [7]:
import random
import time
ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def manhattan_distance(current, goal):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = current[r][c]
            if val != 0:
                target_r, target_c = goal_pos[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def print_matrix(state):
    for row in state:
        print("  " + " ".join(str(x) if x != 0 else "_" for x in row))
    print()


def run_stochastic_hill_climbing(start, goal, max_steps=1000):
    current = start
    path = []
    steps_count = 0
    curr_h = manhattan_distance(current, goal)

    while current != goal:
        if steps_count >= max_steps:
            return None, steps_count

        x, y = find_zero(current)
        better_neighbors = []

        # Lọc ra tất cả các hướng đi làm giảm Heuristic
        for move, (dx, dy), _ in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                neighbor = swap(current, x, y, nx, ny)
                neighbor_h = manhattan_distance(neighbor, goal)
                if neighbor_h < curr_h:
                    better_neighbors.append((neighbor, neighbor_h, move))

        if better_neighbors:
            # Chọn ngẫu nhiên 1 trong các hướng tốt
            current, curr_h, move = random.choice(better_neighbors)
            path.append(move)
            steps_count += 1
        else:
            return None, steps_count

    return path, steps_count

if __name__ == "__main__":

    START_STATE = ((1, 2, 3),
                   (4, 0, 6),
                   (7, 5, 8))

    GOAL_STATE  = ((1, 2, 3),
                   (4, 5, 6),
                   (7, 8, 0))

    print("Trạng thái Khởi đầu (START):")
    print_matrix(START_STATE)
    print("Trạng thái Đích (GOAL):")
    print_matrix(GOAL_STATE)
    print("-" * 60)

    algorithms = [
        ("Stochastic Hill Climbing", run_stochastic_hill_climbing)
    ]

    for name, algo_func in algorithms:
        start_time = time.time()
        path, nodes_visited = algo_func(START_STATE, GOAL_STATE)
        execution_time = (time.time() - start_time) * 1000 # tính bằng mili-giây

        print(f"▶ Thuật toán: {name}")
        if path is not None:
            print(f"  - Trạng thái: THÀNH CÔNG ")
            print(f"  - Chi phí đường đi: {len(path)} bước")
            print(f"  - Lộ trình: {' -> '.join(path)}")
        else:
            print(f"  - Trạng thái: THẤT BẠI (Kẹt cục bộ hoặc vượt quá giới hạn)")
        print(f"  - Số vòng lặp thực hiện: {nodes_visited}")
        print(f"  - Thời gian xử lý: {execution_time:.4f} ms")
        print("-" * 60)

Trạng thái Khởi đầu (START):
  1 2 3
  4 _ 6
  7 5 8

Trạng thái Đích (GOAL):
  1 2 3
  4 5 6
  7 8 _

------------------------------------------------------------
▶ Thuật toán: Stochastic Hill Climbing
  - Trạng thái: THÀNH CÔNG 
  - Chi phí đường đi: 2 bước
  - Lộ trình: D -> R
  - Số vòng lặp thực hiện: 2
  - Thời gian xử lý: 0.0000 ms
------------------------------------------------------------


3. Hill Climbing with Random Walks (Leo đồi vượt kẹt ngẫu nhiên)

In [8]:
import random
import time
ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def manhattan_distance(current, goal):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = current[r][c]
            if val != 0:
                target_r, target_c = goal_pos[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def print_matrix(state):
    for row in state:
        print("  " + " ".join(str(x) if x != 0 else "_" for x in row))
    print()


def run_hill_climbing_random_walk(start, goal, max_steps=1000, walk_steps=5):
    current = start
    path = []
    steps_count = 0
    curr_h = manhattan_distance(current, goal)

    while current != goal:
        if steps_count >= max_steps:
            return None, steps_count

        x, y = find_zero(current)
        neighbor_moved = False

        # 1. Thử leo đồi bình thường
        for move, (dx, dy), _ in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                neighbor = swap(current, x, y, nx, ny)
                neighbor_h = manhattan_distance(neighbor, goal)

                if neighbor_h < curr_h:
                    current = neighbor
                    curr_h = neighbor_h
                    path.append(move)
                    steps_count += 1
                    neighbor_moved = True
                    break

        # 2. Nếu kẹt, thực hiện đi ngẫu nhiên một vài bước (Random Walk) để thoát hiểm
        if not neighbor_moved:
            for _ in range(walk_steps):
                if current == goal:
                    break
                x, y = find_zero(current)
                valid_moves = []
                for move, (dx, dy), _ in ACTIONS:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < 3 and 0 <= ny < 3:
                        valid_moves.append((move, dx, dy))

                if valid_moves:
                    move, dx, dy = random.choice(valid_moves)
                    current = swap(current, x, y, x + dx, y + dy)
                    curr_h = manhattan_distance(current, goal)
                    path.append(move)
                    steps_count += 1

    return path, steps_count

if __name__ == "__main__":

    START_STATE = ((1, 2, 3),
                   (4, 0, 6),
                   (7, 5, 8))

    GOAL_STATE  = ((1, 2, 3),
                   (4, 5, 6),
                   (7, 8, 0))

    print("Trạng thái Khởi đầu (START):")
    print_matrix(START_STATE)
    print("Trạng thái Đích (GOAL):")
    print_matrix(GOAL_STATE)
    print("-" * 60)

    algorithms = [
        ("Hill Climbing with Random Walks", run_hill_climbing_random_walk)
    ]

    for name, algo_func in algorithms:
        start_time = time.time()
        path, nodes_visited = algo_func(START_STATE, GOAL_STATE)
        execution_time = (time.time() - start_time) * 1000 # tính bằng mili-giây

        print(f"▶ Thuật toán: {name}")
        if path is not None:
            print(f"  - Trạng thái: THÀNH CÔNG ")
            print(f"  - Chi phí đường đi: {len(path)} bước")
            print(f"  - Lộ trình: {' -> '.join(path)}")
        else:
            print(f"  - Trạng thái: THẤT BẠI (Kẹt cục bộ hoặc vượt quá giới hạn)")
        print(f"  - Số vòng lặp thực hiện: {nodes_visited}")
        print(f"  - Thời gian xử lý: {execution_time:.4f} ms")
        print("-" * 60)

Trạng thái Khởi đầu (START):
  1 2 3
  4 _ 6
  7 5 8

Trạng thái Đích (GOAL):
  1 2 3
  4 5 6
  7 8 _

------------------------------------------------------------
▶ Thuật toán: Hill Climbing with Random Walks
  - Trạng thái: THÀNH CÔNG 
  - Chi phí đường đi: 2 bước
  - Lộ trình: D -> R
  - Số vòng lặp thực hiện: 2
  - Thời gian xử lý: 0.0000 ms
------------------------------------------------------------


4. Local Beam Search (Tìm kiếm chùm cục bộ)

In [9]:
import time
ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def manhattan_distance(current, goal):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = current[r][c]
            if val != 0:
                target_r, target_c = goal_pos[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def print_matrix(state):
    for row in state:
        print("  " + " ".join(str(x) if x != 0 else "_" for x in row))
    print()


def run_local_beam_search(start, goal, max_steps=1000, k=3):
    curr_h = manhattan_distance(start, goal)
    # Cấu trúc chùm: (giá_trị_h, trạng_thái_hiện_tại, đường_đi_tới_đây)
    beam = [(curr_h, start, [])]
    steps_count = 0

    while steps_count < max_steps:
        # Kiểm tra xem có nhánh nào chạm đích chưa
        for _, state, path in beam:
            if state == goal:
                return path, steps_count

        successors = []
        # Sinh tất cả các trạng thái con từ toàn bộ các node trong chùm hiện tại
        for _, state, path in beam:
            x, y = find_zero(state)
            for move, (dx, dy), _ in ACTIONS:
                nx, ny = x + dx, y + dy
                if 0 <= nx < 3 and 0 <= ny < 3:
                    neighbor = swap(state, x, y, nx, ny)
                    neighbor_h = manhattan_distance(neighbor, goal)
                    successors.append((neighbor_h, neighbor, path + [move]))

        if not successors:
            return None, steps_count

        # Sắp xếp theo Heuristic tốt nhất và chỉ giữ lại `k` node đứng đầu
        successors.sort(key=lambda x: x[0])
        beam = successors[:k]
        steps_count += 1

    return None, steps_count

if __name__ == "__main__":

    START_STATE = ((1, 2, 3),
                   (4, 0, 6),
                   (7, 5, 8))

    GOAL_STATE  = ((1, 2, 3),
                   (4, 5, 6),
                   (7, 8, 0))

    print("Trạng thái Khởi đầu (START):")
    print_matrix(START_STATE)
    print("Trạng thái Đích (GOAL):")
    print_matrix(GOAL_STATE)
    print("-" * 60)

    algorithms = [
        ("Local Beam Search (k=3)", run_local_beam_search)
    ]

    for name, algo_func in algorithms:
        start_time = time.time()
        path, nodes_visited = algo_func(START_STATE, GOAL_STATE)
        execution_time = (time.time() - start_time) * 1000 # tính bằng mili-giây

        print(f"▶ Thuật toán: {name}")
        if path is not None:
            print(f"  - Trạng thái: THÀNH CÔNG ")
            print(f"  - Chi phí đường đi: {len(path)} bước")
            print(f"  - Lộ trình: {' -> '.join(path)}")
        else:
            print(f"  - Trạng thái: THẤT BẠI (Kẹt cục bộ hoặc vượt quá giới hạn)")
        print(f"  - Số vòng lặp thực hiện: {nodes_visited}")
        print(f"  - Thời gian xử lý: {execution_time:.4f} ms")
        print("-" * 60)

Trạng thái Khởi đầu (START):
  1 2 3
  4 _ 6
  7 5 8

Trạng thái Đích (GOAL):
  1 2 3
  4 5 6
  7 8 _

------------------------------------------------------------
▶ Thuật toán: Local Beam Search (k=3)
  - Trạng thái: THÀNH CÔNG 
  - Chi phí đường đi: 2 bước
  - Lộ trình: D -> R
  - Số vòng lặp thực hiện: 2
  - Thời gian xử lý: 0.0000 ms
------------------------------------------------------------
